In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [4]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model="gpt-4.1-nano")

In [5]:
# Creating custom tools with Langchain
## Defining an add function
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): First integer to be added
        b (int): second interger to be added
        
    Return:
        int: sum of a and b
    """
    
    return a + b

In [6]:
tools = [add]

llm_with_tools = llm.bind_tools(tools)

In [7]:
# Subtract tool
@tool
def subtract(a: int, b: int) -> int:
    """
        Subtract b from a.
    """
    return a - b

In [8]:
# Multiply tool 
@tool
def multiply(a: int, b: int) -> int:
    """
        Multiply a and b.
    """
    return a * b

In [9]:
tools = [add, subtract, multiply]

tool_map = {tool.name: tool for tool in tools}

llm_with_tools = llm.bind_tools(tools)

In [10]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

In [11]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

In [12]:
print(type(response_1))
print(response_1)

<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'tool_calls': [{'id': 'call_GUSE8MlfvlhCq09vc0H8PaSA', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'add'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 133, 'total_tokens': 150, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--019c66a6-22e7-7921-9158-6e13d019b9f8-0' tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 2}, 'id': 'call_GUSE8MlfvlhCq09vc0H8PaSA', 'type': 'tool_call'}] usage_metadata={'input_tokens': 133, 'output_tokens': 17, 'total_tokens': 150, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0,

In [13]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

tool name:
add
tool args:
{'a': 3, 'b': 2}
tool call ID:
call_GUSE8MlfvlhCq09vc0H8PaSA


In [14]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='call_GUSE8MlfvlhCq09vc0H8PaSA'


In [15]:
chat_history.append(tool_message)

In [16]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
3 + 2 equals 5.


In [17]:
# Building an agent

class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map
        
    def run(self, query: str) -> str:
        # Step 1: Initial User message
        chat_history = [HumanMessage(content=query)]
        
        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        
        if not response.tool_calls:
            return response.content  # Direct response, no tool needed
        
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]
        
        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)
        
        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [18]:
my_agent = ToolCallingAgent(llm)

print(my_agent.run("one plus 2"))

print(my_agent.run("one - 2"))

print(my_agent.run("three times two"))

One plus two equals three.
The result of \(1 - 2\) is \(-1\).
Three times two equals six.
